In [2]:
import os
import sqlite3
import pandas as pd
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma


# VECTORIZE

## ADDRESS PREP (DATA SOURCE 1)

In [2]:
location_mukim_district_state = pd.read_csv('../output/malaysia-postcodes-location-mukim-district-state.csv',dtype='string')
location_mukim_district_state['address'] = location_mukim_district_state.apply(
    lambda row: f"{row['location']}, {row['mukim']}, {row['postcode']}, {row['district']}, {row['state']}", axis=1
) 

location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: x.split(', '))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: [i for i in x if i != '<NA>'])
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: list(dict.fromkeys(x)))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: ', '.join(x))
address=location_mukim_district_state[['address']]
address.to_csv('../output/address_src_1.csv', index=False)

## ADDRESS PREP (DATA SOURCE 2)

In [3]:
locations_osm = pd.read_csv('../data_source/processed_my_osm.csv',dtype='string')
locations_osm['address'] = locations_osm['address'].str.upper()
address = locations_osm[['address']]
address.to_csv('../output/address_src_2.csv', index=False)

## ADDRESS PREP (DATA SOURCE 3)

In [4]:
locations_school = pd.read_csv('../data_source/school.csv',dtype='string')
locations_school = locations_school[['ALAMATSURAT','POSKODSURAT','BANDARSURAT','NEGERI']]

locations_school['address'] = locations_school.apply(
    lambda row: f"{row['ALAMATSURAT']}, {row['POSKODSURAT']}, {row['BANDARSURAT']}, {row['NEGERI']}", axis=1
) 

address = locations_school[['address']]
address.to_csv('../output/address_src_3.csv', index=False)

## ADDRESS PREP (DATA SOURCE 4)

In [3]:
import pandas as pd

# Read all 4 yellowpages CSV files
yellowpages_files = [
    '../data_source/yellowpages_1.csv',
    '../data_source/yellowpages_2.csv',
    '../data_source/yellowpages_3.csv',
    '../data_source/yellowpages_4.csv'
]

all_addresses = []
for file in yellowpages_files:
    df = pd.read_csv(file, dtype='string')
    # Extract Address column, drop NaN, strip whitespace
    addresses = df['Address'].dropna().astype(str).str.strip().str.upper()
    # Replace newlines with spaces and clean up multiple spaces
    addresses = addresses.str.replace('\n', ' ').str.replace('\r', ' ').str.replace(r'\s+', ' ', regex=True)
    all_addresses.append(addresses)

# Combine all addresses
combined_addresses = pd.concat(all_addresses, ignore_index=True)
# Remove duplicates
combined_addresses = combined_addresses.drop_duplicates()

# Create dataframe and save
address_df = pd.DataFrame({'address': combined_addresses})
address_df.to_csv('../output/address_src_4.csv', index=False)
print(f"Saved {len(address_df)} unique addresses from yellowpages to ../output/address_src_4.csv")

Saved 427394 unique addresses from yellowpages to ../output/address_src_4.csv


## ADDRESS EMBEDDING

#### Source 1

In [8]:
file_path_csv = '../output/address_src_1.csv'
import pandas as pd
from langchain_core.documents import Document

# load CSV via pandas
addr_df = pd.read_csv(file_path_csv, dtype='string')
addresses = addr_df['address'].dropna().astype(str).tolist()
documents1 = [Document(page_content=addr, metadata={'source': file_path_csv}) for addr in addresses]

documents1[:2]
# Keep a quick preview for debugging

[Document(metadata={'source': '../output/address_src_1.csv'}, page_content='ABI, 01000, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address_src_1.csv'}, page_content='ARAU, 02600, PERLIS')]

#### Source 2

In [9]:
file_path_csv = '../output/address_src_2.csv'
import pandas as pd
from langchain_core.documents import Document

# load CSV via pandas
addr_df = pd.read_csv(file_path_csv, dtype='string')
addresses = addr_df['address'].dropna().astype(str).tolist()
documents2 = [Document(page_content=addr, metadata={'source': file_path_csv}) for addr in addresses]

documents2[:2]
# Keep a quick preview for debugging

[Document(metadata={'source': '../output/address_src_2.csv'}, page_content='433, JALAN 5/46, PETALING JAYA, 46000, PETALING, SELANGOR'),
 Document(metadata={'source': '../output/address_src_2.csv'}, page_content='433, JALAN 5/46, PETALING JAYA, 46000, PETALING JAYA, SELANGOR')]

### Source 3

In [10]:
file_path_csv = '../output/address_src_3.csv'
import pandas as pd
from langchain_core.documents import Document

# load CSV via pandas
addr_df = pd.read_csv(file_path_csv, dtype='string')
addresses = addr_df['address'].dropna().astype(str).tolist()
documents3 = [Document(page_content=addr, metadata={'source': file_path_csv}) for addr in addresses]

documents3[:2]
# Keep a quick preview for debugging

[Document(metadata={'source': '../output/address_src_3.csv'}, page_content='JALAN KELAB, 35000, TAPAH, PERAK'),
 Document(metadata={'source': '../output/address_src_3.csv'}, page_content='JALAN TAPAH ROAD, 35400, TAPAH ROAD, PERAK')]

### Source 4

In [11]:
file_path_csv = '../output/address_src_4.csv'
import pandas as pd
from langchain_core.documents import Document

# load CSV via pandas
addr_df = pd.read_csv(file_path_csv, dtype='string')
addresses = addr_df['address'].dropna().astype(str).tolist()
documents4 = [Document(page_content=addr, metadata={'source': file_path_csv}) for addr in addresses]

documents4[:2]
# Keep a quick preview for debugging

[Document(metadata={'source': '../output/address_src_4.csv'}, page_content='NO 24, JALAN PUSAT BCH 1/3, BANDAR COUNTRY HOMES , RAWANG, SELANGOR'),
 Document(metadata={'source': '../output/address_src_4.csv'}, page_content='B2-G-03, JALAN DOKTOR U1/67, TEMASYA 8 , GLENMARIE, SELANGOR')]

## CREATE DOCUMENT

#### Source 1

In [ ]:
persist_directory = "../output/vectorstore"


In [ ]:
import numpy as np

# Reuse the source-1 documents created earlier in the notebook.
documents1 = [
    Document(page_content=doc.page_content.replace('address: ', ''), metadata=dict(doc.metadata))
    for doc in documents1
]

# Update metadata using the source-1 address shape.
for doc in documents1:
    address_parts = doc.page_content.split(', ')
    doc.metadata['state'] = address_parts[-1] if len(address_parts) >= 1 else np.nan
    doc.metadata['district'] = address_parts[-2] if len(address_parts) >= 2 else np.nan
    doc.metadata['postcode'] = address_parts[-3] if len(address_parts) >= 3 else np.nan
    doc.metadata['city'] = np.nan

# documents1

In [11]:
documents1[0]

Document(metadata={'source': '../output/address_src_1.csv', 'row': 0, 'state': 'PERLIS', 'district': 'KANGAR', 'postcode': '01000', 'city': nan}, page_content='ABI, 01000, KANGAR, PERLIS')

In [12]:
documents1[10000].page_content[:1000]  # Display the first 1000 characters of the first document


'LORONG TALANG, 13600, PERAI, PULAU PINANG'

In [13]:
print(len(documents1))
total_docs  = len(documents1)

58394


In [14]:
from sentence_transformers import SentenceTransformer
from langchain.embeddings import HuggingFaceEmbeddings

# Initialize the embedding
oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="llama3.2:latest") # 3072-dim
hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim

c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\izardy\AppData\Local\Temp\ipykernel_37696\2698455389.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim


In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding (384-dim, matches the persisted vectorstore)
hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
'''
for i in tqdm(range(0, len(documents))):
    vectorstore = Chroma.from_documents(
        documents=[documents[i]], 
        embedding=oembed, 
        persist_directory=persist_directory,
        collection_name="base_address"  # Specify the collection name here
    )
print('Data Ingested into Vectorstore')
'''

In [16]:
chunk_size = 100
for i in tqdm(range(0, len(documents1), chunk_size)):
    batch = documents1[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


  0%|          | 0/584 [00:00<?, ?it/s]c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 584/584 [19:15<00:00,  1.98s/it]


#### Source 2

In [ ]:
import numpy as np

# Reuse the source-2 documents created earlier in the notebook.
documents2 = [
    Document(page_content=doc.page_content.replace('address: ', ''), metadata=dict(doc.metadata))
    for doc in documents2
]

# Update metadata using the source-2 address shape.
for doc in documents2:
    address_parts = doc.page_content.split(', ')
    doc.metadata['state'] = address_parts[-1] if len(address_parts) >= 1 else np.nan
    doc.metadata['district'] = address_parts[-2] if len(address_parts) >= 2 else np.nan
    doc.metadata['postcode'] = address_parts[-3] if len(address_parts) >= 3 else np.nan
    doc.metadata['city'] = address_parts[-4] if len(address_parts) >= 4 else np.nan

# documents2

In [19]:
documents2[0].page_content[:1000] 

'433, JALAN 5/46, PETALING JAYA, 46000, PETALING, SELANGOR'

In [20]:
print(len(documents2))
total_docs  = len(documents2)

65395


In [21]:
# Define the folder path for Chroma's in-memory storage
persist_directory = "../output/vectorstore"

In [22]:
chunk_size = 100
for i in tqdm(range(0, len(documents2), chunk_size)):
    batch = documents2[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


  0%|          | 0/654 [00:00<?, ?it/s]c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 654/654 [22:44<00:00,  2.09s/it]


### Source 3

In [ ]:
import numpy as np

# Reuse the source-3 documents created earlier in the notebook.
documents3 = [
    Document(page_content=doc.page_content.replace('address: ', ''), metadata=dict(doc.metadata))
    for doc in documents3
]

# Update metadata using the source-3 address shape.
for doc in documents3:
    address_parts = doc.page_content.split(', ')
    doc.metadata['state'] = address_parts[-1] if len(address_parts) >= 1 else np.nan
    doc.metadata['district'] = address_parts[-2] if len(address_parts) >= 2 else np.nan
    doc.metadata['postcode'] = address_parts[-3] if len(address_parts) >= 3 else np.nan
    doc.metadata['city'] = np.nan

# documents3

In [26]:
documents3[1000].page_content[:1000] 

'9 JALAN KELAB, 31000, BATU GAJAH, PERAK'

In [27]:
print(len(documents3))
total_docs  = len(documents3)

10201


In [28]:
chunk_size = 100
for i in tqdm(range(0, len(documents3), chunk_size)):
    batch = documents3[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


100%|██████████| 103/103 [03:30<00:00,  2.04s/it]
